# Bifurcation-induced tipping: May harvesting

EWS (AR1, variance, ResDMD) on **b_harvesting** trajectories. No control parameter for bifurcation-induced tipping, so ResDMD with control is omitted (same analysis as rate-induced examples but without control).

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kews" / "__init__.py").is_file())
NOTEBOOK_DIR = REPO_ROOT / "examples"
for path in (REPO_ROOT, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUTPUT_DIR = REPO_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from experiments.indicators import rolling_variance, rolling_acf, spectral_gap_series


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from experiments.systems import generate
import kews
from kews.observables import time_delay, rff, mlp

LABEL_FONTSIZE = 16
LEGEND_FONTSIZE = 14
TICK_FONTSIZE = 13


In [ ]:
n_samples = 50
num_timesteps = 500
burnin_idx = 0

def extract_1d(traj):
    """Extract 1D state from trajectory (May harvesting is 1D biomass; the simulator returns (T,2) with second column 0)."""
    if traj.ndim > 1:
        return traj[:, 0]
    return traj

# Generate tipping trajectories (bifurcation-induced: no control used for EWS)
tipping_trajectories_viz = []
tipping_full_params = []  # for optional viz of h(t)
tipping_indices = []
key = jax.random.PRNGKey(42)

for i in range(n_samples):
    key, subkey = jax.random.split(key)
    full_result, part_result, info = generate(
        subkey,
        tipping_type='b_harvesting',
        sigma_tilde=1.0,
        null_simulation=False,
        num_timesteps=num_timesteps
    )
    tipping_trajectories_viz.append(extract_1d(full_result.trajectory[burnin_idx:]))
    tipping_full_params.append(np.array(full_result.param_values[burnin_idx:], dtype=np.float64))
    tipping_indices.append(info.get('transition_index', None))

# Generate non-tipping (null) trajectories
non_tipping_trajectories_viz = []
non_tipping_full_params = []

for i in range(n_samples):
    key, subkey = jax.random.split(key)
    full_result, part_result, info = generate(
        subkey,
        tipping_type='b_harvesting',
        sigma_tilde=1.0,
        null_simulation=True,
        num_timesteps=num_timesteps
    )
    non_tipping_trajectories_viz.append(extract_1d(full_result.trajectory[burnin_idx:]))
    non_tipping_full_params.append(np.array(full_result.param_values[burnin_idx:], dtype=np.float64))

# Universal cutoff (earliest transition index) and EWS window
cutoff_idx = min(idx - burnin_idx for idx in tipping_indices if idx is not None)
start_ews = max(0, cutoff_idx - num_timesteps)

# EWS window: [start_ews:cutoff_idx] (same as saddle / rate-induced)
tipping_trajectories = np.array([t[start_ews:cutoff_idx] for t in tipping_trajectories_viz])
non_tipping_trajectories = np.array([t[start_ews:cutoff_idx] for t in non_tipping_trajectories_viz])


In [ ]:
# Stack trajectories: shape = (n_runs, T)
tip_arr = np.asarray(tipping_trajectories_viz)
nontip_arr = np.asarray(non_tipping_trajectories_viz)

x = np.arange(tip_arr.shape[1])

fig, ax = plt.subplots(2, 1, figsize=(6, 8), sharex=True)

# Tipping envelope + mean
ax[0].fill_between(
    x,
    tip_arr.min(axis=0),
    tip_arr.max(axis=0),
    color="C3",
    alpha=0.5,
    linewidth=0,
    label="Tipping trajectories",
)
ax[0].plot(x, tip_arr.mean(axis=0), color="C3", lw=3)

# Non-tipping envelope + mean
ax[0].fill_between(
    x,
    nontip_arr.min(axis=0),
    nontip_arr.max(axis=0),
    color="C0",
    alpha=0.5,
    linewidth=0,
    label="Non-tipping trajectories",
)
ax[0].plot(x, nontip_arr.mean(axis=0), color="C0", lw=3)
ax[0].axvline(cutoff_idx, color="0.4", linestyle="--", lw=2, label="Tipping point")
ax[0].set_ylabel("State", fontsize=LABEL_FONTSIZE)
ax[0].grid(alpha=0.2)
ax[0].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

# Parameter/forcing
ax[1].plot(tipping_full_params[0], color="C3", lw=3, label="Tipping h(t)")
ax[1].plot(non_tipping_full_params[0], color="C0", lw=3, label="Non-tipping h(t)")
ax[1].axhline(
    info["system_config"]["h_bifurcation"],
    color="0.4",
    linestyle="--",
    lw=2,
    label="Threshold",
)
ax[1].set_xlabel("Time", fontsize=LABEL_FONTSIZE)
ax[1].set_ylabel("Parameter", fontsize=LABEL_FONTSIZE)
ax[1].grid(alpha=0.2)
ax[1].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

for a in ax:
    a.spines["top"].set_visible(False)
    a.spines["right"].set_visible(False)
    a.tick_params(labelsize=TICK_FONTSIZE)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/b_harvesting_traj.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# EWS parameters
wL = 0.5  # Rolling window length (fraction of time series)
N_ENSEMBLES = 1 # Number of ensembles for the ResKMD runs



def compute_ews_metrics(trajectories, wL):
    """Compute EWS (AR1, variance, ResDMD) for bifurcation-induced tipping. No control."""
    ar1_series = []
    var_series = []
    resdmd_series = []
    resdmd_mlp_series = []
    spectral_gap_list = []
    for traj in trajectories:
        data = np.asarray(traj)
        ar1 = rolling_acf(data, window=wL)
        var = rolling_variance(data, window=wL)
        spectral_gap = spectral_gap_series(data, wL)

        ## Ensembling for ResKMD runs
        resdmd_ens = []
        resdmd_mlp_ens = []
        for seed in range(N_ENSEMBLES):
            resdmd = kews.resdmd(data, 
                window=wL, 
                observable=rff(n_features=100, scale=0.2, seed=seed)
            )

            resdmd_mlp = kews.resdmd(data, 
                window=wL, 
                observable=mlp(hidden_layers=[16, 8], bottleneck_dim=4, seed=seed), 
                fit_kwargs=dict(
                    lr=1e-3,
                    epochs=1000,
                    batch_size=32,
                    shuffle=True
                ),
            )
            
            resdmd_ens.append(np.array(resdmd))
            resdmd_mlp_ens.append(np.array(resdmd_mlp))
            
        ar1_series.append(np.array(ar1))
        var_series.append(np.array(var))
        resdmd_series.append(np.array(resdmd_ens).mean(axis=0))
        resdmd_mlp_series.append(np.array(resdmd_mlp_ens).mean(axis=0))
        spectral_gap_list.append(spectral_gap)
    return {
        'ar1_series': ar1_series,
        'var_series': var_series,
        'resdmd_series': resdmd_series,
        'resdmd_mlp_series': resdmd_mlp_series,
        'spectral_gap_series': spectral_gap_list,
    }

# Compute EWS on window [start_ews:cutoff_idx]
tipping_ews = compute_ews_metrics(tipping_trajectories, wL)
non_tipping_ews = compute_ews_metrics(non_tipping_trajectories, wL)


In [ ]:
# Plot EWS
start_idx = 0
t = burnin_idx + start_ews + np.arange(len(tipping_ews['ar1_series'][0]))

def masked_array(series_list, start_idx):
    arr = np.array(series_list, dtype=float)
    arr[:, :start_idx] = np.nan
    return arr

def plot_distribution(ax, series_list, color, label, start_idx, band_alpha=0.15):
    arr = masked_array(series_list, start_idx)
    median = np.nanmedian(arr, axis=0)
    q25, q75 = np.nanpercentile(arr, [25, 75], axis=0)
    ax.plot(t, median, color=color, label=label, linewidth=3)
    ax.fill_between(t, q25, q75, color=color, alpha=band_alpha, linewidth=0)

def robust_ylim(series_lists, start_idx, low=2, high=98, margin=0.05):
    """Compute ylim from percentile of valid values so end spikes don't dominate."""
    vals = np.concatenate([masked_array(s, start_idx)[:, start_idx:].ravel() for s in series_lists])
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return 0.0, 1.0
    y0, y1 = np.nanpercentile(vals, [low, high])
    if y0 == y1:
        y0, y1 = y0 - 0.5, y1 + 0.5
    pad = (y1 - y0) * margin
    return y0 - pad, y1 + pad

panel_specs = [
    ('spectral_gap_series', 'Spectral gap', False),
    ('ar1_series', 'AR1', False),
    ('var_series', 'Variance', False),
    ('resdmd_series', 'ResKMD (RFF)', True),
    ('resdmd_mlp_series', 'ResKMD (MLP)', True),
]

fig, axes = plt.subplots(1, len(panel_specs), figsize=(24, 4), sharex=True)

for i, (key, ylabel, use_robust_ylim) in enumerate(panel_specs):
    ax = axes[i]

    plot_distribution(ax, tipping_ews[key], 'C3', 'Tipping', start_idx)
    plot_distribution(ax, non_tipping_ews[key], 'C0', 'Non-tipping', start_idx)

    ax.set_xlabel('Time', fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_xlim([t[0], t[-1]])
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.grid(alpha=0.2)

    if use_robust_ylim:
        ax.set_ylim(robust_ylim([tipping_ews[key], non_tipping_ews[key]], start_idx))

    # only show legend once
    if i == 0:
        ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/b_harvesting_ews.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# AUROC benchmark across trajectories
import pandas as pd
from experiments.metrics import ensemble_roc, plot_auroc, plot_roc

y_true = np.r_[np.ones(len(tipping_ews['ar1_series'])), np.zeros(len(non_tipping_ews['ar1_series']))].astype(int)

result = ensemble_roc(
    y_true,
    {
        'AR1': tipping_ews['ar1_series'] + non_tipping_ews['ar1_series'],
        'Variance': tipping_ews['var_series'] + non_tipping_ews['var_series'],
        'ResKMD (RFF)': tipping_ews['resdmd_series'] + non_tipping_ews['resdmd_series'],
        'ResKMD (MLP)': tipping_ews['resdmd_mlp_series'] + non_tipping_ews['resdmd_mlp_series'],
    },
)

auroc_df = pd.DataFrame(
    [
        {
            'indicator': name,
            'auroc': curve.auroc,
            'n_valid': curve.n_valid,
            'n_missing': curve.n_missing,
        }
        for name, curve in result.curves.items()
    ]
).sort_values('auroc', ascending=False).reset_index(drop=True)
display(auroc_df)

fig, ax = plt.subplots(figsize=(6, 4))
plot_auroc(result, ax=ax, label_fontsize=LABEL_FONTSIZE, tick_labelsize=TICK_FONTSIZE)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/b_harvesting_auroc.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
plot_roc(
    result,
    ax=ax,
    title=None,
    chance=True,
    show_auroc=True,
    linewidth=3,
    bold_top_n=2,
    label_fontsize=LABEL_FONTSIZE,
    tick_labelsize=TICK_FONTSIZE,
    legend_fontsize=LEGEND_FONTSIZE,
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/b_harvesting_roc.png', dpi=300, bbox_inches='tight')
plt.show()
